In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# I-configure ang error mitigation

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** Ang beta release ng bagong execution model ay available na. Ang directed execution model ay nagbibigay ng mas maraming flexibility sa pag-customize ng iyong error mitigation workflow. Tingnan ang gabay na [Directed execution model](/guides/directed-execution-model) para sa karagdagang impormasyon.


<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na requirements.
Inirerekumenda naming gamitin ang mga bersyong ito o mas bago pa.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Ang mga teknik ng error mitigation ay nagpapahintulot sa mga gumagamit na bawasan ang mga error ng Circuit sa pamamagitan ng
pag-model ng ingay ng device sa oras ng execution. Kadalasan itong nagdudulot ng quantum pre-processing overhead na kaugnay ng pagsasanay ng modelo at
classical post-processing overhead para mabawasan ang mga error sa raw na resulta
sa pamamagitan ng paggamit ng nabuong modelo.

Sinusuportahan ng Estimator primitive ang ilang teknik ng error mitigation, kasama ang [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation), [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne), [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec), at [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). Tingnan ang [Mga teknik ng error mitigation at suppression](error-mitigation-and-suppression-techniques) para sa paliwanag ng bawat isa. Kapag gumagamit ng primitives, maaari mong i-on o i-off ang mga indibidwal na paraan. Tingnan ang seksyong [Custom na mga setting ng error](#advanced-error) para sa mga detalye.

> **Note:** Hindi sinusuportahan ng Sampler ang error mitigation, pero maaari mong gamitin ang [mthree](https://qiskit.github.io/qiskit-addon-mthree/) (matrix-free measurement mitigation) package para magsagawa ng error mitigation nang lokal.

Sinusuportahan din ng Estimator ang `resilience_level`. Tinutukoy ng resilience level kung gaano karaming resilience ang dapat itayo laban sa
mga error. Ang mas mataas na antas ay nagdudulot ng mas tumpak na resulta, ngunit mas matagal ang processing time. Maaaring gamitin ang mga resilience level para i-configure ang
kompromiso sa pagitan ng gastos at katumpakan kapag nag-aaplay ng error mitigation sa iyong primitive
query. Binabawasan ng error mitigation ang mga error (bias) sa mga resulta sa pamamagitan ng pagproseso ng
mga output mula sa isang koleksyon, o ensemble, ng mga kaugnay na Circuit. Ang
antas ng pagbawas ng error ay nakasalalay sa paraan na inilapat. Iniaabstrak ng resilience
level ang detalyadong pagpili ng paraan ng error mitigation para payagan ang
mga gumagamit na pag-isipan ang kompromiso sa gastos at katumpakan na angkop sa
kanilang aplikasyon.

Dahil dito, ang bawat antas ay tumutugma sa isang paraan o mga paraan na may
tumataas na antas ng quantum sampling overhead para makapag-eksperimento ka
sa iba't ibang kompromiso sa oras at katumpakan. Ipinapakita ng sumusunod na talahanayan kung
aling mga antas at kaukulang mga paraan ang available para sa bawat isa sa mga
primitive.

> **Info:** Ang error mitigation ay task-specific, kaya ang mga teknik na maaari mong
> i-apply ay nag-iiba depende sa kung nag-sa-sample ka ng distribusyon o nagge-generate ng
> mga expected value.

<span id="resilience-table"></span>

Sinusuportahan ng Estimator ang mga sumusunod na resilience level. Hindi sinusuportahan ng Sampler ang mga resilience level.

| Resilience Level | Kahulugan                                                                                            | Teknik                                                                 |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0                | Walang mitigation                                                                                         | Wala                                                                      |
| 1 [Default]      | Minimal na gastos sa mitigation: Bawasan ang error na kaugnay ng mga readout error               | Twirled Readout Error eXtinction  (TREX) measurement twirling             |
| 2                | Katamtamang gastos sa mitigation. Kadalasang binabawasan ang bias sa mga estimator, ngunit hindi garantisadong zero-bias. | Antas 1 + Zero Noise Extrapolation (ZNE) at gate twirling

> **Info:** Ang mga resilience level ay kasalukuyang nasa beta kaya ang sampling overhead at
> kalidad ng solusyon ay mag-iiba mula circuit patungong circuit. Mga bagong feature,
> advanced na opsyon, at mga tool sa pamamahala ay ilalabas nang tuluy-tuloy.
> Hindi garantisado na maa-apply ang mga tiyak na paraan ng error mitigation sa bawat resilience level.

## I-configure ang Estimator gamit ang mga resilience level
Maaari mong gamitin ang mga resilience level para tukuyin ang mga teknik ng error mitigation, o maaari kang magtakda ng custom na mga teknik nang isa-isa tulad ng inilarawan sa [Custom na mga setting ng error.](#advanced-error)

<details>
<summary>Resilience Level 0</summary>

Walang error mitigation na inilalapat sa programa ng gumagamit.

</details>

<details>
<summary>Resilience Level 1</summary>

Inilalapat ng Antas 1 ang **readout error mitigation** at **measurement twirling** sa pamamagitan ng pag-apply ng model-free na teknik na kilala
bilang Twirled Readout Error eXtinction (TREX). Binabawasan nito ang measurement error
sa pamamagitan ng pag-diagonalize ng noise channel na kaugnay ng measurement sa pamamagitan ng
random na pag-flip ng mga Qubit gamit ang X Gate bago ang measurement. Isang
rescaling term mula sa diagonal noise channel ay natututo sa pamamagitan ng
pag-benchmark ng mga random na Circuit na sinimulan sa zero state. Pinapayagan nito
ang serbisyo na alisin ang bias mula sa mga expected value na nagmumula sa
readout noise. Ang diskarteng ito ay inilarawan nang higit pa sa [Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738).

</details>

<details>
<summary>Resilience Level 2</summary>

Inilalapat ng Antas 2 ang **mga teknik ng error mitigation na kasama sa antas 1** at inilalapat din ang **gate twirling** at gumagamit ng **Zero Noise Extrapolation method (ZNE)**.  Kinukuwenta ng ZNE ang
expected value ng observable para sa iba't ibang noise factor
(amplification stage) at pagkatapos ay ginagamit ang mga nasukat na expected value para
hulaan ang ideal na expected value sa zero-noise limit (extrapolation
stage). Ang diskarteng ito ay may tendensiyang bawasan ang mga error sa expected value, ngunit
hindi garantisadong makaprodyus ng unbiased na resulta.

![Ipinapakita ng larawang ito ang isang graph. Ang x-axis ay may label na Noise amplification factor. Ang y-axis ay may label na Expectation value. Ang isang linya na pataas ang hilig ay may label na Mitigated value. Ang mga puntos malapit sa linya ay mga noise-amplified na value. Mayroong isang pahalang na linya nang kaunti sa itaas ng X-axis na may label na Exact value.](../docs/images/guides/configure-error-mitigation/resilience-2.svg "Ilustrasyon ng ZNE method")

Ang overhead ng pamamaraang ito ay nagpapalaki kasabay ng bilang ng mga noise factor. Ang
mga default na setting ay nag-sa-sample ng expected value sa tatlong noise factor,
na nagdudulot ng halos 3x na overhead kapag ginagamit ang resilience level na ito.

Sa Antas 2, ang TREX method ay random na nagba-flip ng mga Qubit gamit ang X Gate bago ang measurement,
at binabaligtad ang kaukulang measured bit kung nag-apply ng X Gate. Ang diskarteng ito ay inilarawan nang higit pa sa [Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738).

</details>

### Halimbawa
Ang `EstimatorV2` interface ay nagpapahintulot sa mga gumagamit na maayos na gumana sa iba't ibang
paraan ng error mitigation para bawasan ang error sa mga expected value ng
mga observable. Ang sumusunod na code ay gumagamit ng Zero Noise Extrapolation at readout error mitigation sa pamamagitan lamang ng
pagtatakda ng `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Custom na mga setting ng error
Maaari mong i-on at i-off ang mga indibidwal na paraan ng error mitigation at suppression, kasama ang dynamical decoupling, gate at measurement twirling, measurement error mitigation, PEC, at ZNE. Tingnan ang [Mga teknik ng error mitigation at suppression](error-mitigation-and-suppression-techniques) para sa paliwanag ng bawat isa.

> **Note:** - Hindi lahat ng opsyon ay available para sa parehong primitive. Tingnan ang [talahanayan ng available na opsyon](runtime-options-overview#options-table) para sa listahan ng mga available na opsyon.
> - Hindi lahat ng paraan ay gumagana nang magkasama sa lahat ng uri ng Circuit. Tingnan ang [talahanayan ng compatibility ng feature](runtime-options-overview#options-compatibility-table) para sa mga detalye.

<Tabs>
  <TabItem value="Estimator" label="Estimator">
    ```python
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}")
print(f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}")
```
  </TabItem>

  <TabItem value="Sampler" label="Sampler">